<a href="https://colab.research.google.com/github/deruke/aisecops-labs/blob/main/notebooks/Lab1_ML_Phishing_URL_Classification.ipynb" target="_new"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 1: ML for Security -- Phishing URL Classification
## Applying Supervised Machine Learning to Detect Malicious URLs

**Workshop:** Attacking, Defending & Leveraging Agentic AI  
**Authors:** Derek Banks and Dr. Brian Fehrman  
**Time:** 45 minutes  
**Platform:** Google Colab (CPU-only)  
**Theme:** Leverage  

---

### What You Will Learn
- How to apply supervised machine learning to a real cybersecurity problem
- Working with a real-world phishing detection dataset (11,430 URLs, 111 features)
- Feature selection and understanding what URL properties signal phishing
- Training and comparing Random Forest and Logistic Regression classifiers
- Evaluating models with confusion matrices, precision, recall, and F1 scores
- Understanding false positive vs. false negative tradeoffs in security operations

### Prerequisites
- Basic familiarity with Python
- General understanding of what phishing is
- No prior machine learning experience required

### Threat Model Connection
Phishing remains the **#1 initial access vector** in real-world attacks (Verizon DBIR, MITRE ATT&CK T1566).  
Traditional blocklists are reactive -- they only catch *known* malicious URLs.  
Machine learning lets us generalize from known examples to detect *never-before-seen* phishing URLs  
based on their structural properties.

In this lab, we'll use a **real-world dataset** of 11,430 URLs with 111 pre-extracted features --  
the same type of feature engineering used by commercial email gateways, browser safe-browsing  
features, and SOC automation pipelines.

**Dataset:** [Web Page Phishing Detection Dataset](https://www.kaggle.com/datasets/shashwatwork/web-page-phishing-detection-dataset) (Vrbancic et al., 2020)  
**Source:** [GitHub - GregaVrbancic/Phishing-Dataset](https://github.com/GregaVrbancic/Phishing-Dataset)

### Lab Roadmap
1. Environment Setup
2. Background: Why ML for Phishing Detection
3. Loading the Real-World Phishing Dataset
4. Feature Exploration & Understanding
5. Data Exploration & Visualization
6. Model Training
7. Model Evaluation
8. Model Comparison
9. Student Exercises
10. Security Discussion & Key Takeaways

In [ ]:
# ============================================================
# Cell 1: Environment Setup
# ============================================================
# Install and import all required packages.
# This lab runs on CPU only -- no GPU required.
# ============================================================

!pip install -q scikit-learn pandas numpy matplotlib seaborn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    roc_auc_score,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Plot styling
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print('=' * 60)
print('ENVIRONMENT CHECK')
print('=' * 60)
print(f'NumPy      : {np.__version__}')
print(f'Pandas     : {pd.__version__}')
import sklearn; print(f'Scikit-learn: {sklearn.__version__}')
print(f'Random Seed : {RANDOM_SEED}')
print('=' * 60)
print('All packages loaded successfully. Ready to proceed.')
print('=' * 60)

## Section 1: Background -- Why ML for Phishing Detection

### The Problem
Phishing attacks use deceptive URLs to trick users into visiting malicious websites that steal
 credentials, install malware, or exfiltrate data. Defenders face a fundamental challenge:

| Approach | Strength | Weakness |
|----------|----------|----------|
| **Blocklists** | Zero false positives for known URLs | Cannot detect new/unseen phishing URLs |
| **Heuristic Rules** | Fast, interpretable | Brittle; attackers adapt quickly |
| **Machine Learning** | Generalizes to unseen URLs | Requires training data; can be evaded |

### Why URLs Are a Good Feature Source
URLs carry a surprising amount of signal about attacker intent:

- **Legitimate URLs** tend to be short, use well-known domains, and have simple path structures
- **Phishing URLs** often contain excessive special characters, IP addresses instead of domains,
 excessive subdomains, random character strings, and suspicious TLDs

### What We Will Build
A binary classifier that takes **pre-extracted URL features** as input and outputs a prediction:

```
INPUT:  [length_url=45, qty_dot_url=2, domain_in_ip=0, tls_ssl_certificate=1, ...]
OUTPUT: LEGITIMATE (confidence: 0.97)

INPUT:  [length_url=189, qty_dot_url=7, domain_in_ip=1, tls_ssl_certificate=0, ...]
OUTPUT: PHISHING (confidence: 0.94)
```

We will:
1. Load a real-world dataset of 11,430 URLs with 111 pre-extracted features
2. Explore and understand the feature categories
3. Train two classifiers (Random Forest, Logistic Regression)
4. Evaluate their performance with security-relevant metrics

## Section 2: Loading the Real-World Phishing Dataset

We use the **Web Page Phishing Detection Dataset** (Vrbancic et al., 2020), which contains:

- **11,430 URLs** (balanced: ~50% phishing, ~50% legitimate)
- **111 pre-extracted features** across 6 categories
- **Target column:** `phishing` (1 = phishing, 0 = legitimate)

The features were extracted from the URL structure, page content, and external services --
this is the same type of feature engineering used in production phishing detection systems.

We download directly from the dataset's GitHub repository to avoid Kaggle API authentication.

In [ ]:
# ============================================================
# Cell 3: Load the Phishing Dataset
# ============================================================
# Download from GregaVrbancic/Phishing-Dataset on GitHub.
# This is the same dataset hosted on Kaggle by shashwatwork.
# ============================================================
import os
from pathlib import Path

DATASET_URL = "https://raw.githubusercontent.com/GregaVrbancic/Phishing-Dataset/master/dataset_full.csv"
LOCAL_PATH = Path("./data/phishing_dataset.csv")

# Download if not already cached locally
if LOCAL_PATH.exists():
    print(f"Loading cached dataset from {LOCAL_PATH}")
    df = pd.read_csv(LOCAL_PATH)
else:
    print(f"Downloading dataset from GitHub...")
    df = pd.read_csv(DATASET_URL)
    LOCAL_PATH.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(LOCAL_PATH, index=False)
    print(f"Dataset cached to {LOCAL_PATH}")

# Rename target column for consistency throughout the lab
df = df.rename(columns={"phishing": "label"})

print()
print('=' * 60)
print('DATASET LOADED')
print('=' * 60)
print(f'Total URLs:    {len(df)}')
print(f'Features:      {len(df.columns) - 1}')
print(f'Legitimate:    {(df["label"] == 0).sum()}')
print(f'Phishing:      {(df["label"] == 1).sum()}')
print(f'Balance ratio: {(df["label"] == 1).mean():.1%} phishing')
print('=' * 60)

In [ ]:
# Quick look at the data
print("First 5 rows:")
df.head()

In [ ]:
# Check for missing values
missing = df.isnull().sum()
if missing.sum() == 0:
    print("No missing values in the dataset.")
else:
    print("Columns with missing values:")
    print(missing[missing > 0])

## Section 3: Feature Exploration & Understanding

The dataset has **111 pre-extracted features** organized into 6 categories.
Understanding what these features measure is critical -- you can't build a good model
if you don't understand your inputs.

### Feature Categories

| Category | Count | Prefix/Pattern | What They Measure |
|----------|-------|----------------|-------------------|
| **URL Structure** | 18 | `qty_*_url`, `length_url` | Character counts and length of the full URL |
| **Domain** | ~20 | `qty_*_domain`, `domain_length` | Character counts in the domain portion |
| **Directory** | 18 | `qty_*_directory`, `directory_length` | Character counts in the URL path/directory |
| **File** | 18 | `qty_*_file`, `file_length` | Character counts in the filename portion |
| **Parameters** | ~20 | `qty_*_params`, `params_length` | Character counts in query parameters |
| **Network/Security** | ~15 | Various | DNS, TLS, Google index, redirects, etc. |

### Key Intuitions

| Feature | Phishing Signal |
|---------|-----------------|
| `length_url` | Phishing URLs tend to be longer (obfuscation) |
| `qty_dot_url` | More dots = more subdomains = suspicious |
| `domain_in_ip` | Using an IP instead of a domain is a red flag |
| `tls_ssl_certificate` | Legitimate sites more likely to have valid TLS |
| `qty_redirects` | Phishing often uses redirect chains |
| `url_google_index` | Legitimate sites are usually indexed by Google |
| `domain_length` | Very long domains are suspicious |
| `qty_hyphen_url` | Hyphens commonly used to fake domain names |

In [ ]:
# ============================================================
# Cell 5: Explore Feature Categories
# ============================================================
# Group features by their category prefix to understand
# the dataset structure.
# ============================================================

feature_columns = [col for col in df.columns if col != 'label']

# Categorize features by suffix pattern
categories = {
    'URL Structure': [c for c in feature_columns if c.endswith('_url') or c == 'length_url'],
    'Domain': [c for c in feature_columns if c.endswith('_domain') or c in ['domain_length', 'domain_in_ip', 'server_client_domain']],
    'Directory': [c for c in feature_columns if c.endswith('_directory') or c == 'directory_length'],
    'File': [c for c in feature_columns if c.endswith('_file') or c == 'file_length'],
    'Parameters': [c for c in feature_columns if c.endswith('_params') or c in ['params_length', 'qty_params', 'tld_present_params']],
    'Network/Security': [c for c in feature_columns if c in [
        'email_in_url', 'time_response', 'domain_spf', 'asn_ip',
        'time_domain_activation', 'time_domain_expiration', 'qty_ip_resolved',
        'qty_nameservers', 'qty_mx_servers', 'ttl_hostname',
        'tls_ssl_certificate', 'qty_redirects', 'url_google_index',
        'domain_google_index', 'url_shortened'
    ]],
}

print('=' * 60)
print('FEATURE CATEGORIES')
print('=' * 60)
total = 0
for cat_name, cat_features in categories.items():
    print(f'\n  {cat_name} ({len(cat_features)} features):')
    for f in cat_features[:6]:  # Show first 6 per category
        print(f'    - {f}')
    if len(cat_features) > 6:
        print(f'    ... and {len(cat_features) - 6} more')
    total += len(cat_features)

# Check for uncategorized features
categorized = set()
for feats in categories.values():
    categorized.update(feats)
uncategorized = set(feature_columns) - categorized
if uncategorized:
    print(f'\n  Uncategorized ({len(uncategorized)}): {sorted(uncategorized)}')

print(f'\nTotal features: {len(feature_columns)}')
print('=' * 60)

In [ ]:
# Feature statistics
print('Feature statistics (first 20 features):')
df[feature_columns[:20]].describe().round(2)

## Section 4: Data Exploration & Visualization

Before training models, we need to understand our data. Key questions:

1. **Class balance:** Are legitimate and phishing URLs equally represented?
2. **Feature distributions:** Do features differ between classes?
3. **Feature correlations:** Are any features redundant?

In [ ]:
# ============================================================
# Cell 7: Visualize Feature Distributions
# ============================================================
# Compare key features between legitimate and phishing URLs.
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Feature Distributions: Legitimate vs. Phishing', fontsize=16, fontweight='bold')

# Select informative features to visualize
features_to_plot = [
    'length_url', 'qty_dot_url', 'domain_length',
    'qty_hyphen_url', 'qty_redirects', 'tls_ssl_certificate',
]

labels_map = {0: 'Legitimate', 1: 'Phishing'}
colors = {0: '#2ecc71', 1: '#e74c3c'}

for idx, feature in enumerate(features_to_plot):
    ax = axes[idx // 3][idx % 3]
    if feature not in df.columns:
        ax.set_title(f'{feature} (not found)', fontsize=13)
        continue

    for label_val in [0, 1]:
        subset = df[df['label'] == label_val][feature]
        ax.hist(subset, bins=30, alpha=0.6, label=labels_map[label_val],
                color=colors[label_val], edgecolor='white')

    ax.set_title(feature, fontsize=13, fontweight='bold')
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')
    ax.legend()

plt.tight_layout()
plt.show()

# --- Class balance ---
print()
print('=' * 60)
print('CLASS DISTRIBUTION')
print('=' * 60)
class_counts = df['label'].value_counts()
for label_val, count in class_counts.items():
    pct = count / len(df) * 100
    print(f'  {labels_map[label_val]:12s}: {count:5d} ({pct:.1f}%)')
print('=' * 60)

In [ ]:
# ============================================================
# Cell 8: Correlation Heatmap
# ============================================================
# With 111 features, showing the full heatmap is unwieldy.
# We'll focus on the top features most correlated with the
# target label.
# ============================================================

# Compute correlation with target
label_corr = df[feature_columns + ['label']].corr()['label'].drop('label').sort_values()

# Select top 10 positive and top 10 negative correlations
top_positive = label_corr.tail(10).index.tolist()
top_negative = label_corr.head(10).index.tolist()
top_features = top_negative + top_positive

# Plot heatmap of just these top features + label
plt.figure(figsize=(14, 12))
corr_subset = df[top_features + ['label']].corr()
mask = np.triu(np.ones_like(corr_subset, dtype=bool))

sns.heatmap(
    corr_subset,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.5,
)
plt.title('Correlation Heatmap (Top 20 Features by Label Correlation)', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

# Print ranked correlations
print()
print('=' * 60)
print('FEATURE CORRELATION WITH LABEL (Phishing = 1)')
print('=' * 60)
for feature, corr in label_corr.items():
    if abs(corr) >= 0.1:  # Only show features with meaningful correlation
        direction = 'Phishing indicator' if corr > 0 else 'Legitimate indicator'
        bar = '+' * int(abs(corr) * 20)
        print(f'  {feature:35s}  {corr:+.3f}  {bar:20s}  ({direction})')
print('=' * 60)

## Section 5: Model Training

### Train/Test Split
We split the data 80/20: 80% for training, 20% for evaluation.
The test set simulates "URLs we have never seen before" -- this is how we measure
 whether the model *generalizes* or just memorizes.

### Two Models
We train two fundamentally different classifiers:

| Model | Type | Strengths | Weaknesses |
|-------|------|-----------|------------|
| **Random Forest** | Ensemble of decision trees | Handles non-linear relationships, robust to outliers, provides feature importance | Can overfit, less interpretable |
| **Logistic Regression** | Linear model | Fast, interpretable, works well when features are linearly separable | Struggles with complex non-linear patterns |

Comparing both helps us understand whether the decision boundary is linear or more complex.

In [ ]:
# ============================================================
# Cell 10: Train/Test Split and Model Training
# ============================================================

# Separate features (X) and labels (y)
X = df[feature_columns].values
y = df['label'].values

# 80/20 train/test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

print('=' * 60)
print('TRAIN/TEST SPLIT')
print('=' * 60)
print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set:     {X_test.shape[0]} samples')
print(f'Features:     {X_train.shape[1]}')
print(f'Train class distribution: Legitimate={sum(y_train==0)}, Phishing={sum(y_train==1)}')
print(f'Test class distribution:  Legitimate={sum(y_test==0)}, Phishing={sum(y_test==1)}')
print('=' * 60)

# --- Scale features for Logistic Regression ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- Train Random Forest ---
print()
print('Training Random Forest classifier...')
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_split=5,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)
rf_model.fit(X_train, y_train)
rf_train_acc = rf_model.score(X_train, y_train)
rf_test_acc = rf_model.score(X_test, y_test)
print(f'  Train accuracy: {rf_train_acc:.4f}')
print(f'  Test accuracy:  {rf_test_acc:.4f}')

# --- Train Logistic Regression ---
print()
print('Training Logistic Regression classifier...')
lr_model = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_SEED,
    solver='lbfgs',
)
lr_model.fit(X_train_scaled, y_train)
lr_train_acc = lr_model.score(X_train_scaled, y_train)
lr_test_acc = lr_model.score(X_test_scaled, y_test)
print(f'  Train accuracy: {lr_train_acc:.4f}')
print(f'  Test accuracy:  {lr_test_acc:.4f}')

print()
print('=' * 60)
print('TRAINING COMPLETE')
print('=' * 60)

## Section 6: Model Evaluation

### Why Accuracy Is Not Enough
In security, **not all errors are equal:**

| Error Type | ML Term | Security Impact |
|-----------|---------|----------------|
| Phishing URL classified as legitimate | **False Negative** | User visits phishing site, credentials stolen |
| Legitimate URL classified as phishing | **False Positive** | Legitimate email blocked, user frustrated |

### Key Metrics
- **Precision:** Of all URLs flagged as phishing, how many actually were? (Low precision = alert fatigue)
- **Recall:** Of all actual phishing URLs, how many did we catch? (Low recall = missed attacks)
- **F1 Score:** Harmonic mean of precision and recall (balanced metric)
- **ROC/AUC:** How well the model separates classes across all thresholds

In [ ]:
# ============================================================
# Cell 12: Confusion Matrices
# ============================================================

# Generate predictions
rf_preds = rf_model.predict(X_test)
lr_preds = lr_model.predict(X_test_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Random Forest confusion matrix
cm_rf = confusion_matrix(y_test, rf_preds)
ConfusionMatrixDisplay(cm_rf, display_labels=['Legitimate', 'Phishing']).plot(
    ax=axes[0], cmap='Blues', values_format='d'
)
axes[0].set_title('Random Forest', fontsize=14, fontweight='bold')

# Logistic Regression confusion matrix
cm_lr = confusion_matrix(y_test, lr_preds)
ConfusionMatrixDisplay(cm_lr, display_labels=['Legitimate', 'Phishing']).plot(
    ax=axes[1], cmap='Oranges', values_format='d'
)
axes[1].set_title('Logistic Regression', fontsize=14, fontweight='bold')

plt.suptitle('Confusion Matrices on Test Set', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# --- Print detailed interpretation ---
print()
print('=' * 60)
print('CONFUSION MATRIX INTERPRETATION')
print('=' * 60)

for name, cm in [('Random Forest', cm_rf), ('Logistic Regression', cm_lr)]:
    tn, fp, fn, tp = cm.ravel()
    print(f'\n  {name}:')
    print(f'    True Negatives  (correctly identified legitimate): {tn}')
    print(f'    True Positives  (correctly caught phishing):       {tp}')
    print(f'    False Positives (legitimate blocked as phishing):  {fp}  <-- Alert fatigue')
    print(f'    False Negatives (phishing missed as legitimate):   {fn}  <-- DANGER')

print('\n' + '=' * 60)

In [ ]:
# ============================================================
# Cell 13: Classification Reports
# ============================================================

print('=' * 60)
print('CLASSIFICATION REPORT: Random Forest')
print('=' * 60)
print(classification_report(y_test, rf_preds, target_names=['Legitimate', 'Phishing']))

print('=' * 60)
print('CLASSIFICATION REPORT: Logistic Regression')
print('=' * 60)
print(classification_report(y_test, lr_preds, target_names=['Legitimate', 'Phishing']))

In [ ]:
# ============================================================
# Cell 14: ROC Curves
# ============================================================

# Get probability predictions for ROC
rf_probs = rf_model.predict_proba(X_test)[:, 1]
lr_probs = lr_model.predict_proba(X_test_scaled)[:, 1]

# Compute ROC curves
rf_fpr, rf_tpr, _ = roc_curve(y_test, rf_probs)
lr_fpr, lr_tpr, _ = roc_curve(y_test, lr_probs)

# Compute AUC scores
rf_auc = roc_auc_score(y_test, rf_probs)
lr_auc = roc_auc_score(y_test, lr_probs)

# Plot
plt.figure(figsize=(10, 8))
plt.plot(rf_fpr, rf_tpr, color='#3498db', lw=2.5,
         label=f'Random Forest (AUC = {rf_auc:.3f})')
plt.plot(lr_fpr, lr_tpr, color='#e67e22', lw=2.5,
         label=f'Logistic Regression (AUC = {lr_auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Random Chance (AUC = 0.500)')

plt.xlabel('False Positive Rate (Legitimate URLs incorrectly blocked)', fontsize=13)
plt.ylabel('True Positive Rate (Phishing URLs correctly caught)', fontsize=13)
plt.title('ROC Curves: Model Discrimination Ability', fontsize=16, fontweight='bold')
plt.legend(fontsize=12, loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print()
print('=' * 60)
print('ROC / AUC RESULTS')
print('=' * 60)
print(f'  Random Forest AUC:       {rf_auc:.4f}')
print(f'  Logistic Regression AUC: {lr_auc:.4f}')
print()
print('  Interpretation:')
print('    AUC = 1.00 : Perfect classifier')
print('    AUC = 0.50 : No better than random guessing')
print('    AUC > 0.90 : Excellent discrimination')
print('    AUC > 0.80 : Good discrimination')
print('=' * 60)

In [ ]:
# ============================================================
# Cell 15: Feature Importance Analysis
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# --- Random Forest Feature Importance (top 20) ---
rf_importance = pd.Series(rf_model.feature_importances_, index=feature_columns)
rf_top20 = rf_importance.nlargest(20).sort_values(ascending=True)

rf_top20.plot(kind='barh', ax=axes[0], color='#3498db', edgecolor='white')
axes[0].set_title('Random Forest: Top 20 Features', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Importance (Gini)')

# --- Logistic Regression Coefficients (top 20 by magnitude) ---
lr_coefs = pd.Series(lr_model.coef_[0], index=feature_columns)
lr_top20 = lr_coefs.abs().nlargest(20).index
lr_subset = lr_coefs[lr_top20].sort_values(ascending=True)

colors = ['#e74c3c' if c > 0 else '#2ecc71' for c in lr_subset]
lr_subset.plot(kind='barh', ax=axes[1], color=colors, edgecolor='white')
axes[1].set_title('Logistic Regression: Top 20 Coefficients', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Coefficient (positive = phishing indicator)')
axes[1].axvline(x=0, color='black', linestyle='-', linewidth=0.5)

plt.suptitle('Which Features Matter Most?', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print()
print('=' * 60)
print('TOP 10 MOST IMPORTANT FEATURES (Random Forest)')
print('=' * 60)
for feat, importance in rf_importance.sort_values(ascending=False).head(10).items():
    bar = '#' * int(importance * 100)
    print(f'  {feat:35s}  {importance:.4f}  {bar}')
print('=' * 60)

## Section 7: Side-by-Side Model Comparison

In [ ]:
# ============================================================
# Cell 17: Side-by-Side Model Comparison
# ============================================================

# Collect all metrics
comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'AUC'],
    'Random Forest': [
        accuracy_score(y_test, rf_preds),
        precision_score(y_test, rf_preds),
        recall_score(y_test, rf_preds),
        f1_score(y_test, rf_preds),
        rf_auc,
    ],
    'Logistic Regression': [
        accuracy_score(y_test, lr_preds),
        precision_score(y_test, lr_preds),
        recall_score(y_test, lr_preds),
        f1_score(y_test, lr_preds),
        lr_auc,
    ],
})

print('=' * 60)
print('MODEL COMPARISON SUMMARY')
print('=' * 60)
print(comparison.to_string(index=False, float_format='{:.4f}'.format))
print('=' * 60)

# --- Grouped bar chart ---
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(comparison['Metric']))
width = 0.35

bars1 = ax.bar(x - width/2, comparison['Random Forest'], width,
               label='Random Forest', color='#3498db', edgecolor='white')
bars2 = ax.bar(x + width/2, comparison['Logistic Regression'], width,
               label='Logistic Regression', color='#e67e22', edgecolor='white')

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10)

ax.set_ylabel('Score', fontsize=13)
ax.set_title('Model Performance Comparison', fontsize=16, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(comparison['Metric'], fontsize=12)
ax.legend(fontsize=12)
ax.set_ylim(0, 1.1)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# --- Determine winner ---
print()
rf_f1 = f1_score(y_test, rf_preds)
lr_f1 = f1_score(y_test, lr_preds)
winner = 'Random Forest' if rf_f1 >= lr_f1 else 'Logistic Regression'
print(f'Best overall model (by F1 score): {winner}')
print()
print('Note: In a security context, you may prefer the model with higher')
print('RECALL (fewer missed phishing URLs) even if precision is slightly lower.')

In [ ]:
# ============================================================
# Cell 18: Prediction Demo on Individual Test Samples
# ============================================================
# Show how the model classifies individual URLs from the
# held-out test set.
# ============================================================

print('=' * 70)
print('PREDICTION DEMO ON TEST SET SAMPLES')
print('=' * 70)
print()

# Select a mix of correct and potentially interesting predictions
sample_indices = np.random.RandomState(42).choice(len(X_test), size=8, replace=False)

for i, idx in enumerate(sample_indices):
    sample = X_test[idx:idx+1]
    sample_scaled = X_test_scaled[idx:idx+1]
    actual = y_test[idx]

    rf_pred = rf_model.predict(sample)[0]
    rf_conf = rf_model.predict_proba(sample)[0][rf_pred]
    lr_pred = lr_model.predict(sample_scaled)[0]
    lr_conf = lr_model.predict_proba(sample_scaled)[0][lr_pred]

    label_map = {0: 'LEGITIMATE', 1: 'PHISHING'}
    actual_label = label_map[actual]
    rf_label = label_map[rf_pred]
    lr_label = label_map[lr_pred]

    # Show a few key feature values for context
    feat_dict = dict(zip(feature_columns, X_test[idx]))
    key_feats = f"url_len={feat_dict.get('length_url', '?'):.0f}, dots={feat_dict.get('qty_dot_url', '?'):.0f}, domain_ip={feat_dict.get('domain_in_ip', '?'):.0f}, tls={feat_dict.get('tls_ssl_certificate', '?'):.0f}"

    print(f'  Sample {i+1}: [{key_feats}]')
    print(f'    Actual:              {actual_label}')
    print(f'    Random Forest:       {rf_label:12s} (confidence: {rf_conf:.2f})')
    print(f'    Logistic Regression: {lr_label:12s} (confidence: {lr_conf:.2f})')
    rf_match = 'CORRECT' if rf_pred == actual else 'WRONG'
    lr_match = 'CORRECT' if lr_pred == actual else 'WRONG'
    print(f'    RF: {rf_match} | LR: {lr_match}')
    print()

print('=' * 70)

## Section 8: Student Exercises

Now it is your turn. Complete the following exercises to deepen your understanding.

---

### Exercise 1: Feature Selection
The dataset has 111 features, but not all are equally useful. Try training the model
with only the **top 20 features** (by Random Forest importance) and compare performance.

Does reducing features hurt or help? Why might fewer features sometimes improve results?

In [ ]:
# ============================================================
# Exercise 1: Feature Selection
# ============================================================
# TODO: Select the top 20 features by RF importance, retrain
# both models, and compare performance.
#
# Hint:
#   top20_features = rf_importance.nlargest(20).index.tolist()
#   X_train_top20 = df.loc[train_indices, top20_features].values
# ============================================================

# Your code here:


### Exercise 2: Try a Different Classifier
Train an additional classifier. Good options:
- `sklearn.svm.SVC` -- Support Vector Machine
- `sklearn.ensemble.GradientBoostingClassifier` -- Gradient Boosted Trees
- `sklearn.neighbors.KNeighborsClassifier` -- K-Nearest Neighbors

Compare its performance to Random Forest and Logistic Regression.

In [ ]:
# ============================================================
# Exercise 2: Try a Different Classifier
# ============================================================
# TODO: Import a new classifier, train it on the same data,
# and generate a classification report + confusion matrix.
#
# Example:
#   from sklearn.ensemble import GradientBoostingClassifier
#   gb_model = GradientBoostingClassifier(random_state=42)
#   gb_model.fit(X_train, y_train)
#   gb_preds = gb_model.predict(X_test)
#   print(classification_report(y_test, gb_preds, ...))
# ============================================================

# Your code here:


### Exercise 3: Experiment with Train/Test Ratio
What happens to model performance when you change the train/test split?
- Try 60/40, 70/30, 90/10
- Does the model overfit with less training data? Underfit with more?

In [ ]:
# ============================================================
# Exercise 3: Experiment with Train/Test Ratios
# ============================================================
# TODO: Try different test_size values in train_test_split
# and observe how model performance changes.
#
# Hint: Loop over different test_size values, retrain, and
# collect F1 scores. Then plot the results.
# ============================================================

# Your code here:


## Section 9: Security Discussion

### False Positives vs. False Negatives: The Security Tradeoff

In production security systems, the cost of errors is **asymmetric:**

| | Cost of False Positive | Cost of False Negative |
|---|---|---|
| **Email Gateway** | Legitimate email goes to spam. User annoyed. | Phishing email delivered. Credentials stolen. |
| **Browser Warning** | User sees warning on safe site. Clicks through. | User visits phishing site with no warning. |
| **SOC Alert** | Analyst wastes 15 minutes investigating. | Breach goes undetected for weeks. |

In most security contexts, **false negatives are far more dangerous** than false positives.
 This means we typically want to **maximize recall** even at the expense of some precision.

You can adjust this tradeoff by changing the classification **threshold:**
- Default threshold: 0.5 (predict phishing if probability > 50%)
- Lower threshold (e.g., 0.3): Catches more phishing (higher recall) but more false alarms
- Higher threshold (e.g., 0.7): Fewer false alarms but misses more phishing

### Adversarial Evasion

Attackers are not static. Once they know what features we use, they can **adapt:**

| Our Feature | Attacker Adaptation |
|------------|--------------------|
| URL length | Use URL shorteners (bit.ly, tinyurl.com) |
| IP-based domain | Register cheap domains instead |
| Domain length | Use dictionary words in domains |
| Suspicious characters | Move payloads into path or parameters |
| TLS absence | Get free TLS certificates (Let's Encrypt) |

This is the **adversarial ML** problem: the data distribution shifts because attackers
 actively try to evade your model. This is why phishing detection in production requires:
- Continuous retraining on fresh data
- Multiple detection layers (URL + page content + sender reputation)
- Human-in-the-loop for ambiguous cases

### Real-World Deployment Considerations

1. **Latency:** URL classification must happen in milliseconds (before page loads)
2. **Scale:** Billions of URLs per day across an enterprise
3. **Concept drift:** Attacker tactics evolve; models degrade over time
4. **Explainability:** Security analysts need to understand *why* a URL was flagged
5. **False positive management:** Allowlists for known-good domains to reduce noise

## Section 10: Key Takeaways

### What We Learned

1. **Real-world data matters.** We used a dataset of 11,430 real URLs with 111 professionally
 extracted features -- the same kind of features used in production phishing detection systems.

2. **Feature understanding is critical.** With 111 features, knowing which ones matter
 (and why) is more important than having more features. Feature importance analysis
 helps you understand what the model is actually learning.

3. **Different models have different strengths.** Random Forest captures non-linear patterns
 and provides feature importance. Logistic Regression is fast, interpretable, and shows
 directional relationships.

4. **Accuracy is not enough for security.** Precision and recall tell you *how* the model
 fails, not just *how often*. In security, false negatives are usually more costly than
 false positives.

5. **ML is one layer, not the whole solution.** Real-world phishing detection combines
 ML-based URL analysis with blocklists, content inspection, sender reputation, and
 human review.

6. **Attackers adapt.** Models trained on today's phishing URLs will degrade as attackers
 change their techniques. Continuous monitoring and retraining are essential.

### Connection to the Broader Course

This lab demonstrated the **Leverage** theme: using machine learning as a force multiplier
 for defenders. In later labs, we will explore:
- **Attacking AI:** How adversaries can poison training data or evade ML models
- **Defending AI:** How to make ML systems robust against adversarial manipulation
- **Abliteration:** How safety mechanisms in large language models can be bypassed

---

**End of Lab 1**  
**Workshop:** Attacking, Defending & Leveraging Agentic AI  
**Authors:** Derek Banks and Dr. Brian Fehrman  
**Dataset:** [Vrbancic et al., 2020](https://github.com/GregaVrbancic/Phishing-Dataset)